<a href="https://colab.research.google.com/github/henrique68939/ML/blob/main/Python_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [53]:
!pip install llama-index \
             llama-index-llms-google-genai \
             llama-index-vector-stores-chroma \
             llama-index-embeddings-huggingface \
             sentence-transformers \
             chromadb -q

/usr/lib/python3.13/pathlib/_local.py:274: RuntimeWarning: coroutine 'prepare_chat_params' was never awaited
  parsed = [sys.intern(str(x)) for x in rel.split(sep) if x and x != '.']


In [54]:
import nest_asyncio
nest_asyncio.apply()

In [49]:
import nest_asyncio
nest_asyncio.apply()

import os
from google.colab import userdata  # ← importa o gerenciador de secrets
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

# ← busca o valor do Secret pelo nome
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

Settings.llm = GoogleGenAI(model="gemini-3.8-flash")
Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
print("✅ Configurado com segurança!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Configurado!


In [55]:
import os
os.makedirs("data", exist_ok=True)

csv_conteudo = """data_venda,produto,categoria,quantidade,valor_unitario,valor_total,cliente
2024-01-05,Mouse Logitech MX Master 3,Periféricos,2,350.00,700.00,Carlos Silva
2024-01-08,Teclado Mecânico Redragon Kumara,Periféricos,1,280.00,280.00,Ana Souza
2024-01-10,SSD Kingston 480GB,Armazenamento,3,180.00,540.00,Pedro Alves
2024-01-12,Cabo HDMI 2.0 2m,Cabos,5,45.00,225.00,Maria Lima
2024-01-15,Memória RAM 8GB DDR4 Kingston,Memória,4,220.00,880.00,João Costa
2024-01-18,Webcam Logitech C920,Periféricos,1,450.00,450.00,Fernanda Rocha
2024-01-20,HD Externo Seagate 1TB,Armazenamento,2,320.00,640.00,Lucas Mendes
2024-01-22,Mousepad Gamer XL,Periféricos,3,89.00,267.00,Beatriz Santos
2024-01-25,Adaptador USB-C Hub 7 em 1,Adaptadores,2,189.00,378.00,Rafael Oliveira
2024-01-28,Fonte 600W Corsair Bronze,Componentes,1,420.00,420.00,Camila Torres
2024-02-02,Placa de Vídeo GTX 1650,Componentes,1,1200.00,1200.00,Diego Martins
2024-02-05,Monitor LG 24" Full HD,Monitores,1,950.00,950.00,Patricia Nunes
2024-02-07,Cooler CPU 120mm,Componentes,2,120.00,240.00,Thiago Ferreira
2024-02-10,Fone Headset Hyperx Cloud,Periféricos,1,380.00,380.00,Juliana Castro
2024-02-14,Cabo USB-C 1m,Cabos,10,25.00,250.00,André Pereira
2024-02-18,Pendrive Sandisk 64GB,Armazenamento,6,55.00,330.00,Larissa Gomes
2024-02-20,Roteador TP-Link AX1500,Redes,1,350.00,350.00,Bruno Cardoso
2024-02-23,Estabilizador NHS 500VA,Energia,2,280.00,560.00,Melissa Ribeiro
2024-02-27,Teclado Sem Fio Logitech K380,Periféricos,3,210.00,630.00,Gustavo Lopes
2024-03-01,SSD NVMe 1TB WD Black,Armazenamento,2,480.00,960.00,Renata Almeida
"""

with open("data/vendas_loja.csv", "w", encoding="utf-8") as f:
    f.write(csv_conteudo)

print("✅ CSV criado!")

✅ CSV criado!


In [56]:
import chromadb
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore

db = chromadb.PersistentClient(path="./chroma_db")
try:
    db.delete_collection("loja_informatica")
except:
    pass

colecao = db.get_or_create_collection("loja_informatica")
documents = SimpleDirectoryReader("data").load_data()
print("📄 Documentos carregados!")

vector_store = ChromaVectorStore(chroma_collection=colecao)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

print("🔍 Indexando... aguarde")
index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)
print("✅ Indexado!\n")

query_engine = index.as_query_engine()
pergunta = "Quais produtos da categoria Periféricos foram vendidos?"
resposta = query_engine.query(pergunta)
print(f"❓ {pergunta}")
print(f"💡 {resposta}")

📄 Documentos carregados!
🔍 Indexando... aguarde
✅ Indexado!

❓ Quais produtos da categoria Periféricos foram vendidos?
💡 Os produtos vendidos da categoria Periféricos foram:

- Mouse Logitech MX Master 3
- Teclado Mecânico Redragon Kumara
- Webcam Logitech C920
- Mousepad Gamer XL
- Fone Headset Hyperx Cloud
- Teclado Sem Fio Logitech K380


In [52]:
perguntas = [
    "Qual foi a venda de maior valor total?",
    "Quais produtos de armazenamento foram vendidos e por quanto?"
]

for p in perguntas:
    resposta = query_engine.query(p)
    print(f"❓ {p}")
    print(f"💡 {resposta}\n")

❓ Qual foi a venda de maior valor total?
💡 A venda de maior valor total foi a da **Placa de Vídeo GTX 1650**, no valor de **R$ 1.200,00**, realizada para o cliente Diego Martins em 02/02/2024.

❓ Quais produtos de armazenamento foram vendidos e por quanto?
💡 Os produtos da categoria de armazenamento vendidos foram:

* **SSD Kingston 480GB:** 3 unidades com valor unitário de R$ 180,00 (valor total: R$ 540,00)
* **HD Externo Seagate 1TB:** 2 unidades com valor unitário de R$ 320,00 (valor total: R$ 640,00)
* **Pendrive Sandisk 64GB:** 6 unidades com valor unitário de R$ 55,00 (valor total: R$ 330,00)
* **SSD NVMe 1TB WD Black:** 2 unidades com valor unitário de R$ 480,00 (valor total: R$ 960,00)

